# FitMyResume — Final Evaluation: All 4 Methods on the TEST Set

This notebook runs **all four scoring methods** on the held-out **test split** and
produces the headline comparison table for the report.

| Method | What it is |
|--------|-----------|
| BM25 | Keyword overlap (no model) |
| Sentence-Transformer | `all-mpnet-base-v2` cosine similarity |
| Zero-shot Qwen 2.5 7B | Untuned base model |
| Fine-tuned Qwen 2.5 7B (v2) | Our QLoRA distilled model |

**Why one notebook:** all four are scored against the same teacher outputs, on the
same test examples, with the same context length, decoding, and score-extraction
logic — so any difference reflects the *method*, not the setup.

The test split was never used for training or checkpoint selection, so this is the
honest "unseen data" result.

**Important:** the teacher test file only contains scores keyed by `resume_id` /
`job_id` — it does **not** contain the resume/job text. So this notebook first
**joins** the teacher scores with the resume and job text from the processed CSVs
(the same join `build_instruction_corpus.py` does), then runs the four methods.


## 1. Install dependencies

In [1]:
!pip install -q -U \
    'bitsandbytes>=0.45.0' \
    'transformers>=4.46.0' \
    'peft>=0.14.0' \
    'accelerate>=1.1.0' \
    'datasets>=3.1.0' \
    'sentence-transformers' 'rank_bm25' \
    'scikit-learn' 'scipy' 'pandas' 'sentencepiece' 'protobuf'

## 2. Paths and shared config

Everything that must be identical across the methods lives here, in one place.
Edit the paths if your filenames differ — Drive is case-sensitive.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_BASE = '/content/drive/MyDrive/fit-my-resume'

# --- Teacher test outputs (scores keyed by resume_id / job_id) ---
TEACHER_TEST_PATH = f'{DRIVE_BASE}/data/deepseek_teacher_pilot_outputs.jsonl'

# --- Source text for the join (produced by preprocess_data.py) ---
RESUMES_TEST_CSV = f'{DRIVE_BASE}/data/processed/resumes_test.csv'
JOBS_TEST_CSV    = f'{DRIVE_BASE}/data/processed/jobs_test.csv'

# --- Our fine-tuned adapter ---
ADAPTER_PATH = f'{DRIVE_BASE}/models/qwen25-7b-fitmyresume-lora-v2/final'

# --- Output ---
OUT_DIR = f'{DRIVE_BASE}/results'
os.makedirs(OUT_DIR, exist_ok=True)

# --- Shared LLM config (identical for both Qwen models) ---
BASE_MODEL     = 'Qwen/Qwen2.5-7B-Instruct'
MAX_LENGTH     = 6144   # context window (matches training)
MAX_NEW_TOKENS = 512    # room for the full JSON output

print('Teacher test file:', os.path.exists(TEACHER_TEST_PATH))
print('resumes_test.csv: ', os.path.exists(RESUMES_TEST_CSV))
print('jobs_test.csv:    ', os.path.exists(JOBS_TEST_CSV))
print('Adapter dir:      ', os.path.exists(ADAPTER_PATH))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Teacher test file: True
resumes_test.csv:  True
jobs_test.csv:     True
Adapter dir:       True


## 3. Locate the source CSVs (run this if the check above shows False)

If `resumes_test.csv` / `jobs_test.csv` weren't found, this cell searches your
Drive folder for likely matches and prints them, so you can fix the paths above.
Skip this cell if everything above printed `True`.

In [3]:
import glob

print('Searching for resume/job CSVs under', DRIVE_BASE, '...\n')
for pattern in ['**/resumes_*.csv', '**/jobs_*.csv', '**/*test*.csv']:
    hits = glob.glob(f'{DRIVE_BASE}/{pattern}', recursive=True)
    for h in hits:
        print(h)

Searching for resume/job CSVs under /content/drive/MyDrive/fit-my-resume ...

/content/drive/MyDrive/fit-my-resume/data/processed/resumes_test.csv
/content/drive/MyDrive/fit-my-resume/data/processed/jobs_test.csv
/content/drive/MyDrive/fit-my-resume/data/processed/resumes_test.csv
/content/drive/MyDrive/fit-my-resume/data/processed/jobs_test.csv


## 4. Load teacher scores and join with resume/job text

The teacher file gives us `resume_id`, `job_id`, `pairing_strategy`, and the
teacher `score`. We look up the resume and job text by those IDs and build the
same `input` string the model was trained on:
`RESUME:\n<resume>\n\nJOB_DESCRIPTION:\n<job>`.

In [4]:
import json
import pandas as pd

def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

teacher_rows = load_jsonl(TEACHER_TEST_PATH)
print(f'Teacher test rows: {len(teacher_rows)}')
print('Keys per row:', list(teacher_rows[0].keys()))

# Build id -> text lookups from the processed CSVs
def load_text_map(csv_path, id_col, text_col):
    df = pd.read_csv(csv_path)
    assert id_col in df.columns and text_col in df.columns, \
        f'{csv_path} missing {id_col}/{text_col}; has {df.columns.tolist()}'
    return {str(r[id_col]): str(r[text_col]) for _, r in df.iterrows()}

resume_map = load_text_map(RESUMES_TEST_CSV, 'resume_id', 'resume_text')
job_map    = load_text_map(JOBS_TEST_CSV, 'job_id', 'job_description')
print(f'Loaded {len(resume_map)} resumes, {len(job_map)} jobs')

Teacher test rows: 746
Keys per row: ['pair_id', 'resume_id', 'job_id', 'pairing_strategy', 'similarity_score', 'prompt_version', 'model', 'created_at', 'teacher_output']
Loaded 249 resumes, 86 jobs


In [5]:
# Join: produce one clean record per teacher row
examples = []
skipped = 0
for r in teacher_rows:
    rid = str(r['resume_id'])
    jid = str(r['job_id'])
    if rid not in resume_map or jid not in job_map:
        skipped += 1
        continue
    resume_text = resume_map[rid]
    job_text    = job_map[jid]
    examples.append({
        'pair_id':       r['pair_id'],
        'strategy':      r['pairing_strategy'],
        'resume':        resume_text,
        'job':           job_text,
        'input':         f'RESUME:\n{resume_text}\n\nJOB_DESCRIPTION:\n{job_text}',
        'teacher_score': r['teacher_output']['score'],
    })

print(f'Joined examples: {len(examples)}   (skipped {skipped} with no matching text)')
assert len(examples) > 0, 'No examples joined — check that resume_id/job_id match the CSVs.'

import numpy as np
teacher_scores = [e['teacher_score'] for e in examples]
print(f'Teacher score — mean {np.mean(teacher_scores):.1f}, '
      f'range {min(teacher_scores)}-{max(teacher_scores)}')

Joined examples: 746   (skipped 0 with no matching text)
Teacher score — mean 22.0, range 0-88


## 5. The instruction (system prompt) for the fine-tuned model

The fine-tuned model was trained with the v3 teacher prompt as its system message,
so we load that same prompt text. If you don't have the prompt file in Drive, paste
its text into `FINETUNED_SYSTEM` directly.

In [6]:
# Try to read the v3 prompt from the repo; otherwise set it manually.
PROMPT_V3_PATH = f'{DRIVE_BASE}/prompts/teacher_gold_output_prompt_v3.md'

if os.path.exists(PROMPT_V3_PATH):
    with open(PROMPT_V3_PATH, 'r', encoding='utf-8') as f:
        FINETUNED_SYSTEM = f.read().strip()
    print('Loaded v3 prompt from file.')
else:
    # FALLBACK: paste the instruction text your build_instruction_corpus.py used.
    FINETUNED_SYSTEM = (
        'You are a professional resume evaluation assistant. Evaluate the resume '
        'against the job description and return the structured JSON evaluation.'
    )
    print('Prompt file not found — using fallback text. '
          'Edit FINETUNED_SYSTEM to match your training instruction for best accuracy.')

print('\nFirst 200 chars:\n', FINETUNED_SYSTEM[:200])

Loaded v3 prompt from file.

First 200 chars:
 # Teacher Gold Output Prompt v3

You are generating gold training outputs for FitMyResume, a resume-to-job matching and improvement-suggestion system.

Your task is to evaluate one resume against one 


## 6. Shared score-extraction helper

Both LLMs use this exact function to pull the score from their output, so
extraction can never be a source of difference between them.

In [7]:
import re

def extract_score(text):
    m = re.search(r'"score"\s*:\s*(\d+)', text)
    return int(m.group(1)) if m else None

## 7. Baseline 1 — BM25

Keyword overlap between resume and job. No model, no prompt, no context limit, so
it's unaffected by any LLM config.

In [8]:
from rank_bm25 import BM25Okapi
from scipy.stats import pearsonr, spearmanr

def tokenize(text):
    return text.lower().split()

corpus = [tokenize(e['resume']) for e in examples]
bm25 = BM25Okapi(corpus)

bm25_scores = []
for i, e in enumerate(examples):
    scores = bm25.get_scores(tokenize(e['job']))
    bm25_scores.append(float(scores[i]))

pearson_bm25, _  = pearsonr(teacher_scores, bm25_scores)
spearman_bm25, _ = spearmanr(teacher_scores, bm25_scores)
print(f'BM25 — Pearson {pearson_bm25:.3f}  Spearman {spearman_bm25:.3f}')

BM25 — Pearson 0.239  Spearman 0.280


## 8. Baseline 2 — Sentence-Transformer

Cosine similarity of `all-mpnet-base-v2` embeddings, scaled to 0-100. Runs on CPU
to keep the GPU free for the Qwen models.

In [9]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

st_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device='cpu')

print('Encoding resumes...')
resume_embs = st_model.encode([e['resume'] for e in examples],
                              batch_size=32, show_progress_bar=True)
print('Encoding jobs...')
job_embs = st_model.encode([e['job'] for e in examples],
                           batch_size=32, show_progress_bar=True)

st_scores = [
    float(max(0.0, min(100.0,
        cosine_similarity(resume_embs[i:i+1], job_embs[i:i+1])[0][0] * 100)))
    for i in range(len(examples))
]

pearson_st, _  = pearsonr(teacher_scores, st_scores)
spearman_st, _ = spearmanr(teacher_scores, st_scores)
mae_st = np.mean(np.abs(np.array(teacher_scores) - np.array(st_scores)))
print(f'Sentence-Transformer — Pearson {pearson_st:.3f}  '
      f'Spearman {spearman_st:.3f}  MAE {mae_st:.1f}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Encoding resumes...


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Encoding jobs...


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Sentence-Transformer — Pearson 0.493  Spearman 0.544  MAE 21.0


## 9. Load the base Qwen model

The base model loads once. The fine-tuned model (next section) is this same base
with our LoRA adapter applied — so the only difference between the two LLM rows is
the adapter, nothing else: same context length, same greedy decoding, same
extraction.

In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
base_model.eval()
print('Base model loaded.')

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Base model loaded.


In [11]:
# Shared generation: identical decoding + context length for both models.
active_model = None  # set before each run

def generate(messages):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors='pt',
                       truncation=True, max_length=MAX_LENGTH).to(base_model.device)
    with torch.no_grad():
        out = active_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,                 # greedy — identical for both
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                            skip_special_tokens=True)

## 10. Baseline 3 — Zero-shot Qwen (untuned base)

The untuned model gets a plain instruction (it never saw our training prompt).
Both LLMs still see the same resume/job input, context length, decoding, and
extraction.

In [12]:
import time

ZERO_SHOT_SYSTEM = (
    'You are a professional resume evaluation assistant. '
    'Evaluate the resume against the job description and return ONLY valid JSON '
    'with this structure: {"score": <integer 0-100>, "explanation": "<string>"}'
)

active_model = base_model   # plain base model

zero_shot_scores = []
errors = 0
start = time.time()
for i, e in enumerate(examples):
    messages = [
        {'role': 'system', 'content': ZERO_SHOT_SYSTEM},
        {'role': 'user',   'content': e['input']},
    ]
    s = extract_score(generate(messages))
    if s is None:
        errors += 1
    zero_shot_scores.append(s)
    if (i + 1) % 50 == 0 or i == 0:
        el = (time.time() - start) / 60
        print(f'[{i+1}/{len(examples)}]  {el:.1f} min  ok {i+1-errors}/{i+1}')
print(f'\nZero-shot done in {(time.time()-start)/60:.1f} min  |  errors {errors}')

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1/746]  0.1 min  ok 1/1
[50/746]  5.8 min  ok 50/50
[100/746]  11.9 min  ok 100/100
[150/746]  18.0 min  ok 150/150
[200/746]  23.9 min  ok 200/200
[250/746]  29.9 min  ok 250/250
[300/746]  35.9 min  ok 300/300
[350/746]  41.6 min  ok 350/350
[400/746]  47.2 min  ok 400/400
[450/746]  53.2 min  ok 450/450
[500/746]  59.3 min  ok 500/500
[550/746]  64.7 min  ok 550/550
[600/746]  71.0 min  ok 600/600
[650/746]  76.7 min  ok 650/650
[700/746]  82.3 min  ok 700/700

Zero-shot done in 88.0 min  |  errors 0


## 11. Our model — Fine-tuned Qwen v2

Apply the LoRA adapter on top of the same base, then run with the trained
instruction as the system message.

In [13]:
finetuned_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
finetuned_model.eval()

active_model = finetuned_model   # now generate() uses the fine-tuned model

finetuned_scores = []
errors = 0
start = time.time()
for i, e in enumerate(examples):
    messages = [
        {'role': 'system', 'content': FINETUNED_SYSTEM},
        {'role': 'user',   'content': e['input']},
    ]
    s = extract_score(generate(messages))
    if s is None:
        errors += 1
    finetuned_scores.append(s)
    if (i + 1) % 50 == 0 or i == 0:
        el = (time.time() - start) / 60
        print(f'[{i+1}/{len(examples)}]  {el:.1f} min  ok {i+1-errors}/{i+1}')
print(f'\nFine-tuned done in {(time.time()-start)/60:.1f} min  |  errors {errors}')

[1/746]  1.0 min  ok 1/1
[50/746]  44.7 min  ok 50/50
[100/746]  91.2 min  ok 100/100
[150/746]  138.0 min  ok 150/150
[200/746]  184.3 min  ok 200/200
[250/746]  229.0 min  ok 250/250
[300/746]  272.2 min  ok 300/300
[350/746]  317.4 min  ok 350/350
[400/746]  363.3 min  ok 400/400
[450/746]  408.8 min  ok 450/450
[500/746]  455.0 min  ok 500/500
[550/746]  499.1 min  ok 550/550
[600/746]  548.4 min  ok 600/600
[650/746]  591.8 min  ok 650/650
[700/746]  636.3 min  ok 700/700

Fine-tuned done in 678.4 min  |  errors 0


## 12. Final comparison table

In [14]:
def metrics(preds):
    pairs = [(g, p) for g, p in zip(teacher_scores, preds) if p is not None]
    g = [x[0] for x in pairs]; p = [x[1] for x in pairs]
    pr, _ = pearsonr(g, p)
    sr, _ = spearmanr(g, p)
    mae   = np.mean(np.abs(np.array(g) - np.array(p)))
    parse = 100 * len(pairs) / len(preds)
    return pr, sr, mae, parse

zs = metrics(zero_shot_scores)
ft = metrics(finetuned_scores)

print('=' * 74)
print(f'FINAL RESULTS — TEST SET ({len(examples)} examples)')
print('-' * 74)
print(f'{"Method":<28} {"Pearson":>9} {"Spearman":>10} {"MAE":>7} {"Parse%":>8}')
print('-' * 74)
print(f'{"BM25":<28} {pearson_bm25:>9.3f} {spearman_bm25:>10.3f} {"-":>7} {"-":>8}')
print(f'{"Sentence-Transformer":<28} {pearson_st:>9.3f} {spearman_st:>10.3f} {mae_st:>7.1f} {"-":>8}')
print(f'{"Zero-shot Qwen 2.5 7B":<28} {zs[0]:>9.3f} {zs[1]:>10.3f} {zs[2]:>7.1f} {zs[3]:>7.1f}%')
print(f'{"Fine-tuned Qwen v2":<28} {ft[0]:>9.3f} {ft[1]:>10.3f} {ft[2]:>7.1f} {ft[3]:>7.1f}%')
print('=' * 74)

FINAL RESULTS — TEST SET (746 examples)
--------------------------------------------------------------------------
Method                         Pearson   Spearman     MAE   Parse%
--------------------------------------------------------------------------
BM25                             0.239      0.280       -        -
Sentence-Transformer             0.493      0.544    21.0        -
Zero-shot Qwen 2.5 7B            0.635      0.610    15.3   100.0%
Fine-tuned Qwen v2               0.816      0.834     7.0   100.0%


## 13. Per-strategy breakdown (fine-tuned model)

How the model does across the three pairing tiers. Remember `weak_random` pairs are
near-random, so low correlation there is expected, not a failure.

In [15]:
import collections

groups = collections.defaultdict(list)
for e, p in zip(examples, finetuned_scores):
    if p is not None:
        groups[e['strategy']].append((e['teacher_score'], p))

print(f'{"Strategy":<22} {"n":>5} {"MAE":>8} {"Spearman":>10}')
print('-' * 50)
for strat, pairs in sorted(groups.items()):
    g = [x[0] for x in pairs]; p = [x[1] for x in pairs]
    sr, _ = spearmanr(g, p)
    m = np.mean(np.abs(np.array(g) - np.array(p)))
    print(f'{strat:<22} {len(pairs):>5} {m:>8.1f} {sr:>10.3f}')

Strategy                   n      MAE   Spearman
--------------------------------------------------
medium_tfidf             249      6.2      0.753
strong_hybrid            249      9.8      0.815
weak_random              248      5.0      0.744


## 14. Save all predictions for the report

In [16]:
out_path = f'{OUT_DIR}/all_methods_test_results.jsonl'
with open(out_path, 'w', encoding='utf-8') as f:
    for i, e in enumerate(examples):
        f.write(json.dumps({
            'pair_id':         e['pair_id'],
            'strategy':        e['strategy'],
            'teacher_score':   e['teacher_score'],
            'bm25':            bm25_scores[i],
            'sentence_transf': st_scores[i],
            'zero_shot':       zero_shot_scores[i],
            'finetuned_v2':    finetuned_scores[i],
        }) + '\n')
print('Saved:', out_path)

Saved: /content/drive/MyDrive/fit-my-resume/results/all_methods_test_results.jsonl


In [17]:
import json, time

PER_STRATEGY = 4   # 4 per strategy x 3 = 12 full examples (set higher for more)

# Group by strategy, then take the first N from each
by_strat = {}
for e in examples:
    by_strat.setdefault(e['strategy'], []).append(e)

sample = []
for strat, items in by_strat.items():
    sample.extend(items[:PER_STRATEGY])

print(f'Will generate {len(sample)} full outputs '
      f'({PER_STRATEGY} per strategy x {len(by_strat)} strategies)')

active_model = finetuned_model
finetuned_model.eval()

full_path = f'{OUT_DIR}/finetuned_v2_full_outputs_sample.jsonl'
start = time.time()
with open(full_path, 'w', encoding='utf-8') as f:
    for i, e in enumerate(sample):
        text = generate([
            {'role': 'system', 'content': FINETUNED_SYSTEM},
            {'role': 'user',   'content': e['input']},
        ])
        f.write(json.dumps({
            'pair_id':       e['pair_id'],
            'strategy':      e['strategy'],
            'teacher_score': e['teacher_score'],
            'pred_score':    extract_score(text),
            'full_output':   text,
        }) + '\n')
        print(f'[{i+1}/{len(sample)}]  {e["strategy"]}  '
              f'pred={extract_score(text)} teacher={e["teacher_score"]}  '
              f'{(time.time()-start)/60:.1f} min')

print(f'\nSaved {len(sample)} full outputs to {full_path}')

Will generate 12 full outputs (4 per strategy x 3 strategies)
[1/12]  strong_hybrid  pred=65 teacher=45  1.1 min
[2/12]  strong_hybrid  pred=85 teacher=85  2.1 min
[3/12]  strong_hybrid  pred=75 teacher=65  3.2 min
[4/12]  strong_hybrid  pred=25 teacher=25  4.2 min
[5/12]  medium_tfidf  pred=5 teacher=5  5.3 min
[6/12]  medium_tfidf  pred=15 teacher=5  6.2 min
[7/12]  medium_tfidf  pred=55 teacher=45  7.2 min
[8/12]  medium_tfidf  pred=15 teacher=10  8.2 min
[9/12]  weak_random  pred=5 teacher=10  8.8 min
[10/12]  weak_random  pred=15 teacher=12  9.7 min
[11/12]  weak_random  pred=10 teacher=15  10.3 min
[12/12]  weak_random  pred=5 teacher=10  10.9 min

Saved 12 full outputs to /content/drive/MyDrive/fit-my-resume/results/finetuned_v2_full_outputs_sample.jsonl


In [18]:
import json

full_path = f'{OUT_DIR}/finetuned_v2_full_outputs_sample.jsonl'

with open(full_path, 'r', encoding='utf-8') as f:
    rows = [json.loads(line) for line in f if line.strip()]

print(f'{len(rows)} examples saved\n')

for r in rows:
    print('=' * 70)
    print(f"{r['pair_id']}   strategy={r['strategy']}")
    print(f"predicted score: {r['pred_score']}   |   teacher score: {r['teacher_score']}")
    print('-' * 70)
    # full_output is the model's raw JSON string — pretty-print it if it parses
    try:
        parsed = json.loads(r['full_output'])
        print(json.dumps(parsed, indent=2))
    except json.JSONDecodeError:
        print(r['full_output'])      # fall back to raw text
    print()

12 examples saved

test_29483501_job_000234_strong_hybrid   strategy=strong_hybrid
predicted score: 65   |   teacher score: 45
----------------------------------------------------------------------
{"score":65,"explanation":{"matched_qualifications":["Managed over $1 billion in construction projects, demonstrating large-scale budget oversight (resume: 'Managed over $1 billion in construction projects')","Negotiated all A/E and construction contracts, showing contract management experience (resume: 'Negotiated all A/E and construction contracts')","Implemented multiple project delivery method RFPs and contractual documents, aligning with job's RFP preparation (resume: 'Implemented multiple project delivery method RFPs and contractual documents')","Developed and implemented a five-year facility master plan, supporting long-term planning responsibilities (resume: 'Developed and implemented a five-year facility master plan')","Managed capital project budgets and tracked project costs, matc